# HCC Drug Discovery — Full Pipeline (Steps 09–14)
**One notebook to run the complete computational drug discovery pipeline.**

| Step | Module | Input | Output |
|------|--------|-------|--------|
| 09 | PPI Network | `dea_results.csv` | `hub_genes.csv`, `ppi_network.png` |
| 11 | Survival Filter | `hub_genes.csv` | `survival_filtered_genes.csv`, KM plots |
| 12 | Drug–Gene Interactions | `hub_genes.csv` | `dgi_edges_gnn.csv` |
| 13 | GNN Graph Build | `dgi_edges_gnn.csv` | PyG graph in memory |
| 14 | GNN Training & Ranking | graph | `gnn_drug_ranking.csv`, model weights |

**Prerequisites:** Run notebooks 01–04 first, or ensure `dea_results.csv` exists
in `data/processed/`. All paths are resolved automatically from the repo root.


## 0 · Environment setup

In [ ]:
import sys
from pathlib import Path

def _find_repo_root(start):
    for p in [start, *start.parents]:
        if (p / "paths.py").exists():
            return p
    raise FileNotFoundError("paths.py not found — run: python scripts/data_download.py")

REPO_ROOT = _find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from paths import REPO_ROOT, PROC_DIR, FIGURES_DIR, TABLES_DIR, REPORTS_DIR, MODELS_DIR
print(f"Repo root : {REPO_ROOT}")
print(f"Tables    : {TABLES_DIR}")
print(f"Figures   : {FIGURES_DIR}")


In [ ]:
import sys
sys.path.insert(0, str(REPO_ROOT / "scripts"))
from utils import (
    build_ppi_graph, compute_hub_scores, plot_ppi_network,
    query_dgidb, query_chembl, query_opentargets, get_curated_fallback,
    build_gnn_graph, edge_tensors,
    plot_km_grid, plot_cox_forest, plot_drug_ranking,
    plot_training_curves, plot_model_comparison, plot_scatter,
)

import time, io, warnings, pickle
import numpy as np, pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test
import requests
import torch, torch.nn as nn, torch.nn.functional as F
from torch_geometric.nn import GCNConv, GATConv, SAGEConv
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

warnings.filterwarnings("ignore")
print(f"PyTorch : {torch.__version__}")
print("All imports OK")


In [ ]:
# ── Global configuration — edit here if needed ────────────────────────────────
STRING_SCORE  = 400     # STRING confidence threshold (400=medium, 700=high)
LOG2FC_THRESH = 1.0
PADJ_THRESH   = 0.05
TOP_VIZ_NODES = 80      # nodes shown in PPI figure

KM_P_THRESH   = 0.05
COX_P_THRESH  = 0.05
HR_MIN, HR_MAX = 0.8, 1.2
TOP_KM        = 12

W = {"interaction":0.35, "publications":0.20,
     "phase":0.20, "approved":0.15, "hub":0.10}

HIDDEN_DIM   = 128
EMBED_DIM    = 64
DROPOUT      = 0.3
LR           = 0.005
WEIGHT_DECAY = 1e-4
N_EPOCHS     = 300
PATIENCE     = 40
GAT_HEADS    = 4
RANDOM_SEED  = 42

torch.manual_seed(RANDOM_SEED); np.random.seed(RANDOM_SEED)
MODEL_COLORS = {"GCN":"#534AB7","GAT":"#D85A30","GraphSAGE":"#1D9E75"}
DRUG_FEAT_COLS = ["approved","immunotherapy","anti_neoplastic","clinical_phase",
                  "interaction_score","n_publications","source_DGIdb","source_ChEMBL",
                  "source_OpenTargets","type_inhibitor","type_agonist","type_antagonist",
                  "type_antibody","type_binder","type_activator"]
GENE_FEAT_COLS = ["hub_score","survival_target"]
print("Configuration loaded")


---
## Step 09 · PPI Network Analysis
Queries STRING API, builds a NetworkX graph, and ranks hub genes by composite centrality score.


In [ ]:
dea = pd.read_csv(PROC_DIR / "dea_results.csv")
sig = dea[(dea.adj_pvalue<PADJ_THRESH) & (dea.log2FC.abs()>=LOG2FC_THRESH)].copy()
sig["regulation"] = (sig.log2FC>0).map({True:"up",False:"down"})
gene_list = sig.gene.dropna().unique().tolist()
print(f"DEGs: {len(sig)}  ({(sig.regulation=='up').sum()} up, {(sig.regulation=='down').sum()} down)")


In [ ]:
# Query STRING
STRING_URL = "https://string-db.org/api/json/network"
all_edges = []
for i in range(0, len(gene_list), 500):
    batch = gene_list[i:i+500]
    print(f"  Batch {i//500+1}: {len(batch)} genes...")
    try:
        r = requests.post(STRING_URL,
            data={"identifiers":"".join(batch),"species":9606,
                  "required_score":STRING_SCORE,"network_type":"functional",
                  "caller_identity":"hcc_pipeline"}, timeout=60)
        r.raise_for_status(); all_edges.extend(r.json()); print(f"    → {len(r.json())} interactions")
    except Exception as e: print(f"    ✗ {e}")
    time.sleep(1)

keep = {"preferredName_A":"gene_A","preferredName_B":"gene_B","score":"combined_score"}
edges_df = pd.DataFrame(all_edges).rename(columns=keep)[list(keep.values())]
edges_df = edges_df[edges_df.gene_A!=edges_df.gene_B]
edges_df["pair"] = edges_df.apply(lambda r: tuple(sorted([r.gene_A,r.gene_B])),axis=1)
edges_df = edges_df.drop_duplicates("pair").drop(columns="pair")
edges_df["combined_score"] = pd.to_numeric(edges_df.combined_score,errors="coerce")
print(f"Unique interactions: {len(edges_df)}")


In [ ]:
G = build_ppi_graph(sig, edges_df)
hub_df = compute_hub_scores(G, sig)
fig, _ = plot_ppi_network(G, hub_df, top_nodes=TOP_VIZ_NODES,
                          string_score=STRING_SCORE,
                          log2fc_thresh=LOG2FC_THRESH, padj_thresh=PADJ_THRESH)
fig.savefig(FIGURES_DIR/"ppi_network.png", dpi=200, bbox_inches="tight"); plt.show()

hub_cols=["gene","degree","hub_score","deg_c","betweenness","closeness","eigenvector","log2FC","adj_pvalue","regulation"]
hub_df[[c for c in hub_cols if c in hub_df.columns]].to_csv(TABLES_DIR/"hub_genes.csv",index=False)

cyto = edges_df[edges_df.gene_A.isin(G.nodes())&edges_df.gene_B.isin(G.nodes())].copy()
cyto.rename(columns={"gene_A":"source","gene_B":"target","combined_score":"STRING_score"},inplace=True)
cyto.to_csv(TABLES_DIR/"ppi_edges_cytoscape.csv",index=False)

hub_score_map = hub_df.set_index("gene")["hub_score"].to_dict()
print(f"Saved: hub_genes.csv ({len(hub_df)} genes)")
print(f"Top 5: {hub_df.gene.head().tolist()}")


---
## Step 11 · Survival Filter
Downloads TCGA-LIHC data (or falls back to simulation), runs KM + Cox per gene.


In [ ]:
def fetch_tcga_lihc():
    print("Downloading TCGA-LIHC...")
    CLINICAL_URL = ("https://tcga-xena-hub.s3.us-east-1.amazonaws.com/download/"
                    "TCGA.LIHC.sampleMap%2FLIHC_clinicalMatrix")
    EXPR_URL = ("https://tcga-xena-hub.s3.us-east-1.amazonaws.com/download/"
                "TCGA.LIHC.sampleMap%2FHiSeqV2")
    try:
        r = requests.get(CLINICAL_URL,timeout=30); r.raise_for_status()
        clinical = pd.read_csv(io.StringIO(r.text),sep="\t",low_memory=False)
        clinical = clinical.rename(columns={"sampleID":"patient_id","OS.time":"OS_time",
                                             "OS":"OS_event","_OS_IND":"OS_event","_OS":"OS_time"})
        if "patient_id" in clinical.columns:
            clinical = clinical[clinical.patient_id.str[13:15]=="01"].copy()
        clinical = clinical[["patient_id","OS_time","OS_event"]].dropna()
        clinical[["OS_time","OS_event"]] = clinical[["OS_time","OS_event"]].apply(pd.to_numeric,errors="coerce")
        clinical = clinical.dropna()
        r2 = requests.get(EXPR_URL,timeout=120); r2.raise_for_status()
        expr = pd.read_csv(io.StringIO(r2.text),sep="\t",index_col=0,low_memory=False).T.reset_index().rename(columns={"index":"patient_id"})
        expr["patient_id"] = expr.patient_id.str[:15]
        clinical["patient_id"] = clinical.patient_id.str[:15]
        merged = clinical.merge(expr,on="patient_id",how="inner")
        print(f"  ✓ {len(merged)} patients"); return merged, False
    except Exception as e:
        print(f"  ✗ {e} → using simulated data")
        return None, True

def simulate_tcga(genes, n=374):
    np.random.seed(RANDOM_SEED)
    ot=np.random.exponential(800,n).clip(30,3000); oe=np.random.binomial(1,0.55,n)
    expr={g:np.random.randn(n) for g in genes}
    for g in ["APOE","ALB"]:
        if g in expr:
            hi=expr[g]>0; ot[hi]*=np.random.uniform(1.1,1.4,hi.sum()); oe[hi]=np.random.binomial(1,0.40,hi.sum())
    for g in ["XIST","FTL"]:
        if g in expr:
            hi=expr[g]>0; ot[hi]*=np.random.uniform(0.6,0.85,hi.sum()); oe[hi]=np.random.binomial(1,0.70,hi.sum())
    return pd.DataFrame({"patient_id":[f"P{i:04d}" for i in range(n)],
                         "OS_time":ot.clip(30,3000),"OS_event":oe.astype(int),**expr}), True

merged, is_sim = fetch_tcga_lihc()
if is_sim: merged, _ = simulate_tcga(gene_list)
avail = [g for g in gene_list if g in merged.columns]
print(f"Genes with expression: {len(avail)}/{len(gene_list)}")


In [ ]:
results = []
for i,gene in enumerate(avail):
    gd=merged[["OS_time","OS_event",gene]].dropna().copy(); gd.columns=["T","E","expr"]
    if len(gd)<20: continue
    gd["group"]=np.where(gd.expr>=gd.expr.median(),"High","Low")
    hi,lo=gd[gd.group=="High"],gd[gd.group=="Low"]
    if len(hi)<5 or len(lo)<5: continue
    lr=logrank_test(hi.T,lo.T,event_observed_A=hi.E,event_observed_B=lo.E)
    try:
        cd=gd[["T","E","expr"]].copy(); cd["expr"]=(cd.expr-cd.expr.mean())/(cd.expr.std()+1e-9)
        cph=CoxPHFitter(penalizer=0.1); cph.fit(cd,duration_col="T",event_col="E",show_progress=False)
        hr=float(np.exp(cph.params_["expr"]))
        ci_l=float(np.exp(cph.confidence_intervals_.loc["expr","95% lower-bound"]))
        ci_h=float(np.exp(cph.confidence_intervals_.loc["expr","95% upper-bound"]))
        cox_p=float(cph.summary.loc["expr","p"])
    except: hr=ci_l=ci_h=cox_p=np.nan
    results.append({"gene":gene,"logrank_p":lr.p_value,"cox_p":cox_p,"HR":hr,"HR_CI_low":ci_l,"HR_CI_high":ci_h})
    if (i+1)%50==0: print(f"  [{i+1}/{len(avail)}]")

surv_df = pd.DataFrame(results).merge(sig[["gene","log2FC","adj_pvalue","regulation"]],on="gene",how="left")
surv_df = surv_df.sort_values("logrank_p").reset_index(drop=True)

surv_filtered = surv_df[(surv_df.logrank_p<KM_P_THRESH)&(surv_df.cox_p<COX_P_THRESH)&
                         ((surv_df.HR<HR_MIN)|(surv_df.HR>HR_MAX))].copy()
surv_filtered["prognosis"] = surv_filtered.HR.apply(lambda h: "protective" if pd.notna(h) and h<1 else "risk")
surv_genes = set(surv_filtered.gene)
print(f"Significant survival genes: {len(surv_filtered)}")


In [ ]:
top_genes = surv_filtered.head(TOP_KM).gene.tolist()
if len(top_genes)<TOP_KM:
    top_genes += surv_df[~surv_df.gene.isin(top_genes)].head(TOP_KM-len(top_genes)).gene.tolist()

fig = plot_km_grid(top_genes, surv_df, merged, is_simulated=is_sim)
fig.savefig(FIGURES_DIR/"km_plots.png",dpi=200,bbox_inches="tight"); plt.show()

fig, _ = plot_cox_forest(surv_df, top_n=20, cox_p_thresh=COX_P_THRESH)
fig.savefig(FIGURES_DIR/"cox_forest_plot.png",dpi=200,bbox_inches="tight"); plt.show()

surv_df["prognosis"]=surv_df.HR.apply(lambda h: "protective" if pd.notna(h) and h<1 else "risk" if pd.notna(h) else "")
surv_df.to_csv(TABLES_DIR/"survival_results.csv",index=False)
filt_cols=["gene","logrank_p","cox_p","HR","HR_CI_low","HR_CI_high","log2FC","regulation","prognosis"]
surv_filtered[[c for c in filt_cols if c in surv_filtered.columns]].to_csv(TABLES_DIR/"survival_filtered_genes.csv",index=False)
print(f"Saved: survival_filtered_genes.csv ({len(surv_filtered)} genes)")


---
## Step 12 · Drug–Gene Interaction Collection
Queries DGIdb, ChEMBL, and OpenTargets. Falls back to curated dataset if APIs are down.


In [ ]:
print("Querying drug databases...")
all_dgi, apis_ok = [], []

dgidb_e = query_dgidb(hub_df.gene.tolist())
if dgidb_e: all_dgi.extend(dgidb_e); apis_ok.append("DGIdb")

chembl_e = query_chembl(hub_df.gene.tolist())
if chembl_e: all_dgi.extend(chembl_e); apis_ok.append("ChEMBL")

ot_e = query_opentargets(hub_df.gene.tolist())
if ot_e: all_dgi.extend(ot_e); apis_ok.append("OpenTargets")

if not all_dgi:
    all_dgi = get_curated_fallback(hub_df.gene.tolist()); print("Using curated fallback")
else:
    covered = {e["gene"] for e in all_dgi}
    missing = [g for g in hub_df.gene.tolist() if g not in covered]
    all_dgi.extend(get_curated_fallback(missing))

print(f"APIs: {apis_ok or ['curated']}  |  Raw interactions: {len(all_dgi)}")


In [ ]:
dgi_df = pd.DataFrame(all_dgi)
for col in DRUG_FEAT_COLS:
    if col not in dgi_df.columns: dgi_df[col]=0
for col in ["approved","immunotherapy","anti_neoplastic"]:
    dgi_df[col]=dgi_df[col].fillna(False).astype(bool)
dgi_df["interaction_score"]=pd.to_numeric(dgi_df.interaction_score,errors="coerce").fillna(0)
dgi_df["n_publications"]=pd.to_numeric(dgi_df.n_publications,errors="coerce").fillna(0).astype(int)
dgi_df["clinical_phase"]=pd.to_numeric(dgi_df.clinical_phase,errors="coerce").fillna(0).astype(int)
dgi_df["drug"]=dgi_df.drug.str.strip().str.title()
dgi_df["gene"]=dgi_df.gene.str.strip().str.upper()
dgi_df=(dgi_df.sort_values("interaction_score",ascending=False)
              .drop_duplicates(["gene","drug"],keep="first").reset_index(drop=True))

def norm(s): return (s-s.min())/(s.max()-s.min()+1e-9)
dgi_df["composite_score"]=(
    W["interaction"]*norm(dgi_df.interaction_score)+
    W["publications"]*norm(dgi_df.n_publications.clip(0,30))+
    W["phase"]*(dgi_df.clinical_phase/4)+
    W["approved"]*dgi_df.approved.astype(float)+
    W["hub"]*norm(dgi_df.gene.map(hub_score_map).fillna(0))+
    dgi_df.gene.isin(surv_genes).astype(float)*0.10
).clip(0,1).round(4)
dgi_df=dgi_df.sort_values("composite_score",ascending=False).reset_index(drop=True)

# Add GNN feature columns and export
gnn_df=dgi_df.copy()
gnn_df["hub_score"]=gnn_df.gene.map(hub_score_map).fillna(0)
gnn_df["survival_target"]=gnn_df.gene.isin(surv_genes).astype(int)
for src in ["DGIdb","ChEMBL","OpenTargets"]:
    gnn_df[f"source_{src}"]=(gnn_df.source==src).astype(int)
for it in ["inhibitor","agonist","antagonist","antibody","binder","activator"]:
    gnn_df[f"type_{it}"]=(gnn_df.interaction_type.str.lower()==it).astype(int)

gnn_cols=(["gene","drug","composite_score","approved","immunotherapy","anti_neoplastic",
            "clinical_phase","interaction_score","n_publications"]+
           [f"source_{s}" for s in ["DGIdb","ChEMBL","OpenTargets"]]+
           [f"type_{t}" for t in ["inhibitor","agonist","antagonist","antibody","binder","activator"]]+
           ["hub_score","survival_target","interaction_type","directionality","source"])
gnn_df[[c for c in gnn_cols if c in gnn_df.columns]].to_csv(TABLES_DIR/"dgi_edges_gnn.csv",index=False)
print(f"Saved: dgi_edges_gnn.csv ({len(gnn_df)} edges, {gnn_df.gene.nunique()} genes, {gnn_df.drug.nunique()} drugs)")


---
## Steps 13–14 · GNN Graph Build, Training & Drug Ranking


In [ ]:
edges_df = pd.read_csv(TABLES_DIR / "dgi_edges_gnn.csv")
graph_data, node2idx, idx2node, labels, gene_set, drug_set, scaler = build_gnn_graph(edges_df)
feat_dim = graph_data.x.shape[1]

indices = list(range(len(edges_df)))
tr_idx, te_idx = train_test_split(indices, test_size=0.15, random_state=RANDOM_SEED)
tr_idx, va_idx = train_test_split(tr_idx, test_size=0.15/0.85, random_state=RANDOM_SEED)

with open(MODELS_DIR/"feature_scaler.pkl","wb") as f: pickle.dump(scaler,f)
print(f"Graph: {len(node2idx)} nodes, {len(edges_df)} edges, feat_dim={feat_dim}")
print(f"Split: train={len(tr_idx)}, val={len(va_idx)}, test={len(te_idx)}")


In [ ]:
class GCNModel(nn.Module):
    def __init__(self,in_dim,hidden=128,out_dim=64,dropout=0.3):
        super().__init__()
        self.conv1=GCNConv(in_dim,hidden); self.bn1=nn.BatchNorm1d(hidden)
        self.conv2=GCNConv(hidden,out_dim); self.bn2=nn.BatchNorm1d(out_dim)
        self.head=nn.Sequential(nn.Linear(out_dim*2,out_dim),nn.ReLU(),nn.Dropout(dropout),nn.Linear(out_dim,1),nn.Sigmoid())
        self.drop=dropout
    def encode(self,x,ei):
        x=F.relu(self.bn1(self.conv1(x,ei))); x=F.dropout(x,p=self.drop,training=self.training); return F.relu(self.bn2(self.conv2(x,ei)))
    def forward(self,x,ei,src,dst): z=self.encode(x,ei); return self.head(torch.cat([z[src],z[dst]],dim=1)).squeeze(-1)

class GATModel(nn.Module):
    def __init__(self,in_dim,hidden=32,out_dim=64,heads=4,dropout=0.3):
        super().__init__()
        self.conv1=GATConv(in_dim,hidden,heads=heads,dropout=dropout,concat=True); self.bn1=nn.BatchNorm1d(hidden*heads)
        self.conv2=GATConv(hidden*heads,out_dim,heads=1,dropout=dropout,concat=False); self.bn2=nn.BatchNorm1d(out_dim)
        self.head=nn.Sequential(nn.Linear(out_dim*2,out_dim),nn.ELU(),nn.Dropout(dropout),nn.Linear(out_dim,1),nn.Sigmoid()); self.drop=dropout
    def encode(self,x,ei):
        x=F.elu(self.bn1(self.conv1(x,ei))); x=F.dropout(x,p=self.drop,training=self.training); return F.elu(self.bn2(self.conv2(x,ei)))
    def forward(self,x,ei,src,dst): z=self.encode(x,ei); return self.head(torch.cat([z[src],z[dst]],dim=1)).squeeze(-1)

class SAGEModel(nn.Module):
    def __init__(self,in_dim,hidden=128,out_dim=64,dropout=0.3):
        super().__init__()
        self.conv1=SAGEConv(in_dim,hidden,aggr="mean"); self.bn1=nn.BatchNorm1d(hidden)
        self.conv2=SAGEConv(hidden,out_dim,aggr="mean"); self.bn2=nn.BatchNorm1d(out_dim)
        self.head=nn.Sequential(nn.Linear(out_dim*2,out_dim),nn.ReLU(),nn.Dropout(dropout),nn.Linear(out_dim,1),nn.Sigmoid()); self.drop=dropout
    def encode(self,x,ei):
        x=F.relu(self.bn1(self.conv1(x,ei))); x=F.dropout(x,p=self.drop,training=self.training); return F.relu(self.bn2(self.conv2(x,ei)))
    def forward(self,x,ei,src,dst): z=self.encode(x,ei); return self.head(torch.cat([z[src],z[dst]],dim=1)).squeeze(-1)

def _et(idx_list): return edge_tensors(edges_df,idx_list,node2idx,labels)

def train_model(model):
    opt=torch.optim.Adam(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
    sch=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,"min",patience=15,factor=0.5,min_lr=1e-5)
    s_tr,d_tr,y_tr=_et(tr_idx); s_va,d_va,y_va=_et(va_idx)
    best_val,best_state,no_imp=float("inf"),None,0
    history={"train_loss":[],"val_loss":[],"lr":[]}
    for ep in range(1,N_EPOCHS+1):
        model.train(); opt.zero_grad()
        loss=F.mse_loss(model(graph_data.x,graph_data.edge_index,s_tr,d_tr),y_tr)
        loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()
        model.eval()
        with torch.no_grad(): vl=F.mse_loss(model(graph_data.x,graph_data.edge_index,s_va,d_va),y_va).item()
        sch.step(vl); history["train_loss"].append(loss.item()); history["val_loss"].append(vl); history["lr"].append(opt.param_groups[0]["lr"])
        if vl<best_val: best_val=vl; best_state={k:v.clone() for k,v in model.state_dict().items()}; no_imp=0
        else: no_imp+=1
        if no_imp>=PATIENCE: print(f"  Early stop ep {ep}  best_val={best_val:.4f}"); break
        if ep%50==0: print(f"  ep {ep:3d}  train={loss.item():.4f}  val={vl:.4f}")
    model.load_state_dict(best_state); return history, best_val

def evaluate(model,idx_list):
    model.eval(); s,d,yt=_et(idx_list)
    with torch.no_grad(): yp=model(graph_data.x,graph_data.edge_index,s,d).numpy()
    yt=yt.numpy()
    return {"r2":float(r2_score(yt,yp)),"mse":float(mean_squared_error(yt,yp)),"mae":float(mean_absolute_error(yt,yp)),"pred":yp,"true":yt}

print("Models and training functions defined")


In [ ]:
models_cfg={"GCN":GCNModel(feat_dim),"GAT":GATModel(feat_dim),"GraphSAGE":SAGEModel(feat_dim)}
all_results={}
for name,model in models_cfg.items():
    print(f"\n[{name}]")
    history,best_val=train_model(model)
    test_m=evaluate(model,te_idx)
    with torch.no_grad(): embeds=model.encode(graph_data.x,graph_data.edge_index).numpy()
    all_results[name]={"history":history,"best_val":best_val,"test":test_m,"embeddings":embeds,"model":model}
    m=test_m; print(f"  Test → R²={m['r2']:.4f}  MSE={m['mse']:.4f}  MAE={m['mae']:.4f}")

best_name=max(all_results,key=lambda n: all_results[n]["test"]["r2"])
best_model=all_results[best_name]["model"]
print(f"\nBest model: {best_name}  R²={all_results[best_name]['test']['r2']:.4f}")


In [ ]:
fig=plot_training_curves(all_results,best_name)
fig.savefig(FIGURES_DIR/"gnn_training_curves.png",dpi=200,bbox_inches="tight"); plt.show()

fig=plot_model_comparison(all_results)
fig.savefig(FIGURES_DIR/"gnn_model_comparison.png",dpi=200,bbox_inches="tight"); plt.show()

fig=plot_scatter(all_results,best_name)
fig.savefig(FIGURES_DIR/"gnn_predicted_vs_actual.png",dpi=200,bbox_inches="tight"); plt.show()


In [ ]:
best_model.eval()
all_src=torch.tensor([node2idx[g] for g in edges_df.gene])
all_dst=torch.tensor([node2idx[d] for d in edges_df.drug])
with torch.no_grad(): gnn_scores=best_model(graph_data.x,graph_data.edge_index,all_src,all_dst).numpy()

ranking=edges_df.copy()
ranking["gnn_score"]=gnn_scores.round(4)
ranking["original_score"]=ranking.composite_score
ranking["score_delta"]=(ranking.gnn_score-ranking.original_score).round(4)
ranking=ranking.sort_values("gnn_score",ascending=False).reset_index(drop=True)
ranking["rank"]=ranking.index+1

fig,ax=plot_drug_ranking(ranking,best_name=best_name)
fig.savefig(FIGURES_DIR/"gnn_drug_ranking.png",dpi=200,bbox_inches="tight"); plt.show()


## Final export & summary

In [ ]:
# Save model
torch.save(best_model.state_dict(), MODELS_DIR/"gcn_best.pt")

# Node embeddings
embed_df=pd.DataFrame(all_results[best_name]["embeddings"],columns=[f"dim_{i}" for i in range(EMBED_DIM)])
embed_df.insert(0,"node",[idx2node[i] for i in range(len(node2idx))])
embed_df.insert(1,"node_type",["gene" if embed_df.iloc[i].node in gene_set else "drug" for i in range(len(node2idx))])
embed_df.to_csv(TABLES_DIR/"gnn_node_embeddings.csv",index=False)

# Drug ranking
rank_cols=["rank","gene","drug","gnn_score","original_score","score_delta","approved","clinical_phase","interaction_type","source"]
ranking[[c for c in rank_cols if c in ranking.columns]].to_csv(TABLES_DIR/"gnn_drug_ranking.csv",index=False)

print("\n" + "="*55)
print("  Pipeline complete. All outputs saved.")
print("="*55)
outputs = [
    ("hub_genes.csv",                TABLES_DIR),
    ("survival_filtered_genes.csv",  TABLES_DIR),
    ("dgi_edges_gnn.csv",            TABLES_DIR),
    ("gnn_drug_ranking.csv",         TABLES_DIR),
    ("ppi_network.png",              FIGURES_DIR),
    ("km_plots.png",                 FIGURES_DIR),
    ("gnn_drug_ranking.png",         FIGURES_DIR),
    ("gcn_best.pt",                  MODELS_DIR),
]
for fname, folder in outputs:
    path = folder / fname
    status = "✓" if path.exists() else "✗ MISSING"
    size = f"{path.stat().st_size//1024} KB" if path.exists() else ""
    print(f"  {status}  {folder.name}/{fname}  {size}")

print(f"\nTop 10 drug candidates ({best_name}):")
ranking[["rank","drug","gene","gnn_score","approved","clinical_phase"]].head(10)
